# GTEx model building with CLAMP (No FBM Version)

💡 **Environment:** `clamp-analyses`  

This notebook builds latent variable models from GTEx v8 RNA‑seq TPM data using CLAMP. It automates downloading and preprocessing the GTEx matrix, computes an SVD to estimate the model dimension, prepares pathway priors, runs CLAMP (base + full) and saves model outputs (B, Z, summaries) and intermediate files. Configuration and paths are controlled via `config.R`.

## Load libraries

In [1]:
# Create a timestamp to track the start of the analysis
start_time <- Sys.time()
cat("GTEx CLAMP and PLIER analysis started at:", format(start_time), "\n")

GTEx CLAMP and PLIER analysis started at: 2026-01-22 19:51:49 


In [2]:
if (!requireNamespace("PLIER", quietly = TRUE)) {
    devtools::install_github("wgmao/PLIER")
}

# Note: bigstatsr is no longer required for the main workflow
library(data.table)
library(dplyr)
library(rsvd)      # Using rsvd for SVD computation
library(glmnet)
library(Matrix)
library(knitr)
library(here)
library(PLIER)
library(CLAMP)

source(here("config.R"))

set.seed(config$GTEx$RANDOM_SVD_SEED)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Loaded glmnet 4.1-10

here() starts at /home/msubirana/Documents/pivlab/clamp-analyses

Loading required package: RColorBrewer

Loading required package: gplots


---------------------
gplots 3.3.0 loaded:
  * Use citation('gplots') for citation info.
  * Homepage: https://talgalili.github.io/gplots/
  * Report issues: https://github.com/talgalili/gplots/issues
  * Ask questions: https://stackoverflow.com/questions/tagged/gplots
  * Suppress this message with: suppressPackageStartupMessages(library(gplots))
---------------------



Attaching package: ‘gplots’


The following object is masked from ‘package:stats’:

    lowess


Loading required package: pheatmap

Loadin

## Output directory

In [3]:
output_data_dir <- config$GTEx$OUTPUT_DIR
dir.create(output_data_dir, showWarnings = FALSE, recursive = TRUE)

output_data_dir

[1] "/home/msubirana/Documents/pivlab/clamp-analyses/output/gtex"

# Settings

In [4]:
block_size <- config$GENERAL$CHUNK_SIZE
N_CORES    <- config$GTEx$N_CORES

## Download GTEx 

In [5]:
url <- config$GTEx$URL
dest_dir <-  config$GTEx$DATASET_FOLDER
dest_gz  <- file.path(dest_dir, basename(url))

if (!file.exists(dest_gz)) {
  dir.create(dest_dir, recursive = TRUE, showWarnings = FALSE)
  download.file(url, dest_gz, mode = "wb")
  message("Downloaded to: ", dest_gz)
} else {
  message("File already exists, skipping download.")
}

File already exists, skipping download.



## Preprocess GTEx data (In-Memory Version)

In [6]:
exprs_path  <- file.path(config$GTEx$DATASET_FOLDER, 'GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_tpm.gct.gz')
output_file <- config$GTEx$DATASET_FILE

if (!file.exists(output_file)) {
  dir.create(dirname(output_file), recursive = TRUE, showWarnings = FALSE)
  exprs_data <- read.table(exprs_path, header = TRUE, sep = "\t", skip = 2, check.names = FALSE)
  saveRDS(exprs_data, config$GTEx$DATASET_FILE)
  message("File successfully written to: ", config$GTEx$DATASET_FILE)
} else {
  message("Output file already exists. Skipping.")
}

# Aggregate in-place by 'description'
gtex <- readRDS(here(config$GTEx$DATASET_FILE))
gtex <- as.data.table(gtex)
aggregated_gtex <- gtex[, lapply(.SD, sum), by = Description, .SDcols = is.numeric]

genes <- aggregated_gtex$Description
samples <- colnames(aggregated_gtex[, -1])
data_mat <- as.matrix(aggregated_gtex[, -1])
rownames(data_mat) <- genes

cat("Data matrix dimensions: ", dim(data_mat), "\n")

Output file already exists. Skipping.



Data matrix dimensions:  54592 17382 


## Preprocess data

In [7]:
# Preprocess using CLAMP's in-memory function
prep_gtex <- preprocessCLAMP(
  Y = data_mat,
  mean_cutoff = config$GTEx$GENES_MEAN_CUTOFF,
  var_cutoff  = config$GTEx$GENES_VAR_CUTOFF
)

gtex_mat_filt <- prep_gtex$Y_filtered
gtex_rowStats <- prep_gtex$rowStats
gtex_genes <- rownames(gtex_mat_filt)

cat("Filtered matrix dimensions: ", dim(gtex_mat_filt), "\n")
cat("Number of genes after filtering: ", length(gtex_genes), "\n")

Filtered matrix dimensions:  23906 17382 
Number of genes after filtering:  23906 


In [8]:
# Z-score the filtered matrix
gtex_mat_zscore <- zscoreCLAMP(gtex_mat_filt, gtex_rowStats)

message("Z-score transformation complete")
cat("Z-scored matrix dimensions: ", dim(gtex_mat_zscore), "\n")

Z-score transformation complete



Z-scored matrix dimensions:  23906 17382 


In [ ]:
saveRDS(samples, file = file.path(output_data_dir, "gtex_samples_df.rds"))

In [ ]:
saveRDS(gtex_genes, file = file.path(output_data_dir, "gtex_genes_df.rds"))

: 

In [ ]:
saveRDS(gtex_mat_zscore, file = file.path(output_data_dir, "gtex_zscore_df.rds"))

In [ ]:
# Use the z-scored matrix directly
df_gtex_filt <- as.data.frame(gtex_mat_zscore)
head(df_gtex_filt)

,GTEX-1117F-0226-SM-5GZZ7,GTEX-1117F-0426-SM-5EGHI,GTEX-1117F-0526-SM-5EGHJ,GTEX-1117F-0626-SM-5N9CS,GTEX-1117F-0726-SM-5GIEN,GTEX-1117F-1326-SM-5EGHH,GTEX-1117F-2426-SM-5EGGH,GTEX-1117F-2526-SM-5GZY6,GTEX-1117F-2826-SM-5GZXL,GTEX-1117F-2926-SM-5GZYI,⋯,GTEX-ZZPU-1126-SM-5N9CW,GTEX-ZZPU-1226-SM-5N9CK,GTEX-ZZPU-1326-SM-5GZWS,GTEX-ZZPU-1426-SM-5GZZ6,GTEX-ZZPU-1826-SM-5E43L,GTEX-ZZPU-2126-SM-5EGIU,GTEX-ZZPU-2226-SM-5EGIV,GTEX-ZZPU-2426-SM-5E44I,GTEX-ZZPU-2626-SM-5E45Y,GTEX-ZZPU-2726-SM-5NQ8O
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
WASH7P,1.5139414,-0.1112375,1.0449167,2.2783025,-0.2952013,0.3952429,2.5832515,4.2273240,1.9368923,2.7522992,⋯,-0.85869390,-0.63926325,0.3942485,-0.62534167,-0.57694760,-0.05754002,-0.75395057,-0.4440296,-1.1027856,-0.6727413
RP11-34P13.15,-0.2527007,-0.3213043,-0.2957023,-0.2840209,-0.3056420,-0.2718537,-0.2878607,-0.1725267,-0.2799266,-0.2157134,⋯,-0.24994808,-0.16271894,0.4705077,-0.14738271,-0.21589841,0.10317843,-0.12115151,-0.2495780,-0.3104186,-0.1814324
RP11-34P13.16,-0.2290288,-0.3321770,-0.3324351,-0.3106754,-0.3118786,-0.2826366,-0.3222012,-0.1543326,-0.2776653,-0.2462859,⋯,-0.20797205,-0.06203107,0.4016933,-0.07153038,-0.20724377,0.25223758,0.00272252,-0.2519063,-0.3248958,-0.1407170
RP11-34P13.14,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,-0.2911013,0.0207658,-0.1096559,⋯,-0.29110127,-0.29110127,0.6469125,-0.11790180,-0.29110127,-0.29110127,0.17340459,-0.1903863,-0.2911013,0.1463984
RP11-34P13.18,0.6091633,-0.6029156,0.3535757,0.4464296,-0.9576751,-0.6469494,1.7100090,0.9863226,-0.1250529,0.7355213,⋯,-1.02506598,-0.72735903,-0.3119094,-0.93125479,-0.74631272,-0.18555151,-1.05837853,-0.7181694,-1.1631025,-0.5472032
AP006222.2,0.3230707,-0.5338201,-0.2776069,-0.3343933,-0.5886587,-0.6117329,-0.6144298,-0.6695681,-0.3805417,-0.6671708,⋯,-0.01854688,1.15898496,0.9776879,-0.24749060,-0.03607726,1.98456079,-0.36915441,-0.1495003,-0.2660698,3.3300546


In [ ]:
saveRDS(df_gtex_filt, file = file.path(output_data_dir, "df_gtex_filt_df.rds"))
write.csv(df_gtex_filt, file = file.path(output_data_dir, "df_gtex_filt_df.csv"))

: 

## SVD computation using rsvd

Using the `rsvd` package for randomized SVD

In [ ]:
if (!file.exists(file.path(output_data_dir, "gtex_svdRes_df.rds"))) {
  
  n_genes   <- nrow(gtex_mat_zscore)
  n_samples <- ncol(gtex_mat_zscore)
  SVD_K_gtex <- min(n_genes, n_samples) - 1
  
  message("Using SVD K = ", SVD_K_gtex)
  message("Computing SVD using rsvd package...")
  
  # Use rsvd for randomized SVD computation
  gtex_svdRes <- rsvd::rsvd(
    A = gtex_mat_zscore,
    k = SVD_K_gtex,
  )
  
  message("SVD computation complete")
  saveRDS(gtex_svdRes, file = file.path(output_data_dir, "gtex_svdRes_df.rds"))

} else {
  message("gtex_svdRes_df already exists, skipping SVD computation.")
}

Using SVD K = 17381

Computing SVD using rsvd package...



## Estimate K for CLAMP

In [ ]:
CLAMP_K_gtex <- num.pc(list(d = gtex_svdRes$d)) * 2
message("Inferred CLAMP K = ", CLAMP_K_gtex)

In [ ]:
saveRDS(CLAMP_K_gtex, file = file.path(output_data_dir, "CLAMP_K_gtex_df.rds"))

write.csv(
  as.data.frame(CLAMP_K_gtex),
  file = file.path(output_data_dir, "CLAMP_K_gtex_df.csv"),
  row.names = TRUE
)

## CLAMPbase

In [ ]:
gtex_baseRes <- CLAMPbase(
  Y      = gtex_mat_zscore,
  svdres = gtex_svdRes,
  clamp_k = CLAMP_K_gtex,
  trace  = TRUE
)

In [ ]:
gtex_baseRes$Z <- data.frame(gtex_baseRes$Z)
rownames(gtex_baseRes$Z) <- gtex_genes
head(gtex_baseRes$Z)

gtex_baseRes$B <- data.frame(gtex_baseRes$B)
colnames(gtex_baseRes$B) <- samples
head(gtex_baseRes$B)

In [ ]:
saveRDS(gtex_baseRes, file = file.path(output_data_dir, "CLAMPbase_df.rds"))

In [ ]:
model_dir <- file.path(output_data_dir, "CLAMPbase_df")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_baseRes$B
write.csv(B, file.path(model_dir, "B_df.csv"))

Z <- gtex_baseRes$Z
rownames(Z) <- gtex_genes
write.csv(Z, file.path(model_dir, "Z_df.csv"))

## Prepare pathway priors

In [ ]:
gtex_gmtList <- list(
  KEGG = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=KEGG_2021_Human"),
  BP = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GO_Biological_Process_2025"),
  GTEx_Tissues = getGMT("https://maayanlab.cloud/Enrichr/geneSetLibrary?mode=text&libraryName=GTEx_Tissues_V8_2023")
)

# prefix each gene‐set name with its library to guarantee uniqueness
for(lib in names(gtex_gmtList)) {
  names(gtex_gmtList[[lib]]) <- paste0(lib, "_", names(gtex_gmtList[[lib]]))
}

gtex_pathMat <- gmtListToSparseMat(gtex_gmtList)
gtex_matched <- getMatchedPathwayMat(gtex_pathMat, gtex_genes)
gtex_chatObj <- getChat(gtex_matched)

## CLAMPfull

In [ ]:
gtex_fullRes <- CLAMPfull(
    Y = gtex_mat_zscore,
    priorMat = as.matrix(gtex_matched),
    clamp.base.result = gtex_baseRes,
    svdres = gtex_svdRes,
    clamp_k = CLAMP_K_gtex,
    doCrossval = TRUE,
    trace = TRUE
)

In [ ]:
gtex_baseRes$Z <- data.frame(gtex_baseRes$Z)
rownames(gtex_baseRes$Z) <- gtex_genes
head(gtex_baseRes$Z)

gtex_baseRes$B <- data.frame(gtex_baseRes$B)
colnames(gtex_baseRes$B) <- samples
head(gtex_baseRes$B)

gtex_fullRes$summary <- gtex_fullRes$summary %>%
    dplyr::rename(LV = LV_index)  %>% 
    dplyr::mutate(LV = paste0('LV', LV))

In [ ]:
saveRDS(gtex_fullRes, file = file.path(output_data_dir, "CLAMPfull_df.rds"))

In [ ]:
head(gtex_fullRes$B)
dim(gtex_fullRes$B)

In [ ]:
model_dir <- file.path(output_data_dir, "CLAMPfull_df")
dir.create(model_dir, showWarnings = FALSE, recursive = TRUE)

B <- gtex_fullRes$B
colnames(B) <- samples
write.csv(B, file.path(model_dir, "B_df.csv"))

Z <- gtex_fullRes$Z
rownames(Z) <- gtex_genes
write.csv(Z, file.path(model_dir, "Z_df.csv"))

summary <- gtex_fullRes$summary
write.csv(summary, file.path(model_dir, "summary_df.csv"))